In [15]:
from pathlib import Path
from datasets import load_dataset
import pandas as pd
import json
from collections import defaultdict

In [16]:
def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / ".git").exists():
            return path
    raise FileNotFoundError(f"No .git folder found above {start}")
        
ROOT = find_repo_root()
DATA_DIR = ROOT / "data"
CORPUS_PATH = DATA_DIR / "corpus" / "pooled_corpus.parquet"
CORPUS_PATH.parent.mkdir(parents=True, exist_ok=True)
STATS_PATH = ROOT / "data" / "corpus" / "corpus_stats.json"
COVERAGE_PATH = ROOT / "results" / "corpus" / "coverage.json"
COVERAGE_PATH.parent.mkdir(parents=True, exist_ok=True)

print(ROOT) 

/mnt/data/Programing/NLP_cw


In [17]:
# Load Dataset for questions
hotpot_ds = load_dataset("hotpotqa/hotpot_qa", "distractor")

# Load Pooled Corpus
corpus = pd.read_parquet(CORPUS_PATH)

# Load Corpus sizes statistics
with open(STATS_PATH) as f:
    size_stats = json.load(f)

In [18]:
# Check if Corpus PIDs are unique
assert corpus.pid.is_unique
# Print Corpus Table
corpus

,pid,title,sentence_list,text
0,d71835e9bb51ce07,Radio City (Indian radio station),[Radio City is India's first private FM radio ...,Radio City is India's first private FM radio s...
1,ae47cb81312e1d3d,History of Albanian football,[Football in Albania existed before the Albani...,Football in Albania existed before the Albania...
2,e9f35a68b8def30f,Echosmith,"[Echosmith is an American, Corporate indie pop...","Echosmith is an American, Corporate indie pop ..."
3,5e7fab9a040f929d,Women's colleges in the Southern United States,[Women's colleges in the Southern United State...,Women's colleges in the Southern United States...
4,6f056c505396dea1,First Arthur County Courthouse and Jail,"[The First Arthur County Courthouse and Jail, ...","The First Arthur County Courthouse and Jail, w..."
...,...,...,...,...
509310,8b1b428f1a451ecd,PowerPC e200,[The e200 core is developed from the MPC5xx fa...,The e200 core is developed from the MPC5xx fam...
509311,8b6843127aa799fa,OMAP,[OMAP (Open Multimedia Applications Platform) ...,OMAP (Open Multimedia Applications Platform) i...
509312,49e40889b08b008e,Zet (hardware),[Zet is a clone x86 processor where its machin...,Zet is a clone x86 processor where its machine...
509313,f1e7bb861e862e77,Xetal,[Xetal is the name of a family of non commerci...,Xetal is the name of a family of non commercia...


## Titles
Check titles with more then 1 occurence in the corpus

In [19]:
print(f"The Pooled Corpus contains {corpus.title.nunique()} unique titles\n")

# Get titles that appear more then once
title_count = corpus.title.value_counts()
duplicate_titles = title_count[title_count > 1]
print(f"Titles that appear more then once:\n {duplicate_titles}\n")

# Number of titles that appear more then once
print(f"Count of titles that appear more then once: {duplicate_titles.count()}\n")

# Maximum number of texts for one title
print(f"The maximum number of texts for a title are: {duplicate_titles.max()}")



The Pooled Corpus contains 507494 unique titles

Titles that appear more then once:
 title
The Hunger Games (novel)                      2
750 7th Avenue                                2
Seventh Avenue (Manhattan)                    2
Dragostea Din Tei                             2
The Green Hornet (2011 film)                  2
                                             ..
Ethiopian Airlines accidents and incidents    2
Black Book of the Admiralty                   2
Van Andel Institute                           2
Richard Thompson (sprinter)                   2
Doris Lessing                                 2
Name: count, Length: 1821, dtype: int64

Count of titles that appear more then once: 1821

The maximum number of texts for a title are: 2


### Analise all rows with titles that appear more then once
- Pairs with identical text after replacing \xa0: 1821 / 1821
- Pairs with identical sentence lists after replacing \xa0: 1821 / 1821

In [20]:
identical_txt_count = 0
equal_sl_count = 0

for i in range(duplicate_titles.count()):
    t = duplicate_titles.index[i]
    entries = corpus[corpus.title == t]
    a_txt,b_txt  = entries.text
    
    # Are these two texts identical if we ignore the difference between non-breaking spaces and normal spaces?
    if a_txt.replace("\xa0", " ") == b_txt.replace("\xa0", " "):
        identical_txt_count += 1

    a_sl, b_sl = entries.sentence_list

    if len(a_sl) == len(b_sl):
        equal_sl_count += 1


print(f"There are {identical_txt_count} pairs of texts that are identical if we ignore the difference between non-breaking spaces and normal spaces")
print(f"There are {equal_sl_count} paris of sentence list that have the same size")

There are 1821 pairs of texts that are identical if we ignore the difference between non-breaking spaces and normal spaces
There are 1821 paris of sentence list that have the same size


In [21]:
NBSP = "\xa0"

def normalise(s: str) -> str:
    return s.replace(NBSP, " ")

# Filter once, then loop over the groups
duplicates = corpus[corpus.title.isin(duplicate_titles.index)]

identical_txt_count = 0
identical_sl_count = 0
example = None

for title, group in duplicates.groupby("title"):
    a_txt, b_txt = group.text
    a_sl, b_sl = group.sentence_list

    identical_txt_count += normalise(a_txt) == normalise(b_txt)

    # Same sentences in the same order, so sent_id means the same in both versions
    identical_sl_count += [normalise(x) for x in a_sl] == [normalise(x) for x in b_sl]
        
    # Keep the first differing sentence pair as evidence
    if example is None:
        example = next(((title, x, y) for x, y in zip(a_sl, b_sl) if x != y), None)

n_pairs = len(duplicate_titles)
print(f"Pairs with identical text after replacing \\xa0: {identical_txt_count} / {n_pairs}")
print(f"Pairs with identical sentence lists after replacing \\xa0: {identical_sl_count} / {n_pairs}")

title, x, y = example
print(f"\nExample: {title}\n  A: {x!r}\n  B: {y!r}")

Pairs with identical text after replacing \xa0: 1821 / 1821
Pairs with identical sentence lists after replacing \xa0: 1821 / 1821

Example: &quot;V&quot; Is for Vagina
  A: '"V" Is for Vagina is the debut studio album by Maynard James Keenan\'s side project Puscifer, released on  30, 2007 (2007--) .'
  B: '"V" Is for Vagina is the debut studio album by Maynard James Keenan\'s side project Puscifer, released on \xa030,\xa02007\xa0(2007--) .'


**Important**: The example title is stored as &quot;V&quot; *(html version)* Is for Vagina, while its text has the real characters "V" 

### Lenghts

In [22]:
words_per_paragraph = corpus.text.str.split().str.len()
sentence_per_paragraph = corpus.sentence_list.str.len()

wpp_data = words_per_paragraph.describe(percentiles=[0.50, 0.95, 0.99])
spp_data = sentence_per_paragraph.describe(percentiles=[0.50, 0.95, 0.99])

print("Words per Paragraph Data")
print(wpp_data, "\n")


print("Sentence per Paragraph Data")
print(spp_data)


Words per Paragraph Data
count    509315.000000
mean         78.410828
std          43.654496
min           3.000000
50%          70.000000
95%         159.000000
99%         219.000000
max        1378.000000
Name: text, dtype: float64 

Sentence per Paragraph Data
count    509315.000000
mean          3.716588
std           2.044170
min           1.000000
50%           3.000000
95%           7.000000
99%          10.000000
max          85.000000
Name: sentence_list, dtype: float64


In [23]:
SPLITS = ["train", "validation"]

questions = {
    split: hotpot_ds[split].select_columns(["id", "type", "level", "supporting_facts"]).to_pandas() for split in SPLITS
}

for split, q in questions.items():
    print(f"=== {split}: {len(q):,} questions ===")

    # Type x level (with totals)
    print(pd.crosstab(q.type, q.level, margins=True), "\n")

    # Gold paragraphs = distinct titles; supporting sentences = (title, sent_id) pairs
    q["n_gold_titles"] = q.supporting_facts.map(lambda sf: len(set(sf["title"])))
    q["n_sup_sents"] = q.supporting_facts.map(lambda sf: len(sf["title"]))
    
    print("Gold titles per question:")
    print(q.n_gold_titles.value_counts().sort_index().to_string(), "\n")
    print("Supporting sentences per question:")
    print(q.n_sup_sents.describe(percentiles=[0.5, 0.95]).round(2).to_string(), "\n")


=== train: 90,447 questions ===
level        easy   hard  medium    All
type                                   
bridge      14466  12451   46074  72991
comparison   3506   3210   10740  17456
All         17972  15661   56814  90447 

Gold titles per question:
n_gold_titles
2    90447 

Supporting sentences per question:
count    90447.00
mean         2.38
std          0.67
min          2.00
50%          2.00
95%          4.00
max         12.00 

=== validation: 7,405 questions ===
level       hard   All
type                  
bridge      5918  5918
comparison  1487  1487
All         7405  7405 

Gold titles per question:
n_gold_titles
2    7405 

Supporting sentences per question:
count    7405.00
mean        2.43
std         0.71
min         2.00
50%         2.00
95%         4.00
max         8.00 



In [24]:
# title -> sentence counts of its version(s) in the corpus (a list: 1,821 titles have two versions)
title_to_lens = defaultdict(list)
for title, sentence_list in zip(corpus.title, corpus.sentence_list):
    title_to_lens[title].append(len(sentence_list))

coverage = {}
for split, q in questions.items():
    n_titles = n_titles_found = 0
    n_facts = n_facts_valid = 0
    bad_questions = set()
    bad_examples = []

    for qid, sf in zip(q.id, q.supporting_facts):
        # Title coverage: is each gold paragraph in the corpus?
        for title in set(sf["title"]):
            n_titles += 1
            n_titles_found += title in title_to_lens
        
        for title, sent_id in zip(sf["title"], sf["sent_id"]):
            n_facts += 1
            lens = title_to_lens.get(title, [])
            if any(sent_id < n for n in lens):
                n_facts_valid += 1
            else:
                bad_questions.add(qid)
                bad_examples.append((qid, title, int(sent_id), lens))


    coverage[split] = {
            "gold_titles": n_titles,
            "gold_titles_found": n_titles_found,
            "supporting_facts": n_facts,
            "supporting_facts_valid": n_facts_valid,
            "questions_with_invalid_fact": len(bad_questions),
        }   

    print(f"=== {split} ===")
    print(f"Gold titles found:       {n_titles_found:,} / {n_titles:,}")
    print(f"Supporting facts valid:  {n_facts_valid:,} / {n_facts:,}")
    print(f"Questions with an invalid fact: {len(bad_questions)}")
    for qid, title, sent_id, lens in bad_examples[:3]:
        print(f"  {qid}  {title!r}  sent_id={sent_id}  paragraph has {lens} sentence(s)")
    print()


    with open(COVERAGE_PATH, "w") as f:
        json.dump(coverage, f, indent=2)


=== train ===
Gold titles found:       180,894 / 180,894
Supporting facts valid:  215,662 / 215,684
Questions with an invalid fact: 22
  5a7b23ca554299042af8f703  'Minoru Suzuki'  sent_id=2  paragraph has [2] sentence(s)
  5abed6d45542990832d3a0ef  'Jonathan Taylor Thomas'  sent_id=20  paragraph has [2] sentence(s)
  5ab6b2fb5542995eadef0060  'Francis Poulenc'  sent_id=3  paragraph has [3] sentence(s)

=== validation ===
Gold titles found:       14,810 / 14,810
Supporting facts valid:  18,004 / 18,005
Questions with an invalid fact: 1
  5ae61bfd5542992663a4f261  'Jimmy Butler (basketball)'  sent_id=902  paragraph has [5] sentence(s)

